# MMGTFFF — Experiment 1: Exact StockNet Protocol A Reproduction
### Roadmap Priority 1 / 2: "Fix StockNet split + labels + trading-day alignment" then establish a trustworthy benchmark.

Run top-to-bottom in a fresh Google Colab runtime. Self-contained — clones `stocknet-dataset` only.

**This is a faithful port of the actual training-data pipeline in the official
[`stocknet-code`](https://github.com/yumoxu/stocknet-code) repo**
(`src/DataPipe.py`, `src/config.yml`), not a re-derivation from our own project's
preprocessing conventions. Every rule below was read directly out of that repo's
source, not guessed:

| Rule | Source | Value |
|---|---|---|
| Universe | `config.yml: stocks` | exact 88 tickers (9 sectors) |
| Split (end-exclusive) | `config.yml: dates` | train `2014-01-01→2015-08-01`, dev `2015-08-01→2015-10-01`, test `2015-10-01→2016-01-01` |
| Window | `config.yml: max_n_days=5` | **calendar**-anchored: target minus 4 calendar days |
| Target label | `DataPipe._get_mv_class` | `1` if `mv>0` else `0`, computed on `ProsusAI`-independent raw movement % |
| Buffer-zone filter | `DataPipe._get_prices_and_ts` | discard as a *target* if `-0.5% <= mv < 0.55%` — applied **only** to the day being predicted, never to input days |
| Price features | `DataPipe._get_prices(data)` = `data[3:6]` | the 3 official per-day price ratios in `price/preprocessed/{TICKER}.txt` (StockNet's own precomputed values, not ours) |
| Tweet alignment | `DataPipe._trading_day_alignment` | every calendar day's tweets attach to the **first actual trading day strictly after it** (never same-day) |

### The one thing this notebook does differently from the official code, and why

The official window length **T is variable** — a calendar-anchored 4-day lookback
naturally contains fewer real trading days around weekends/holidays (e.g. a
Tuesday target's lookback spans Fri–Mon, giving only 2 real trading days, not 4).
The official TensorFlow code pads short windows with zeros and tracks the true
length in a separate `T` tensor for masking. We do exactly the same here — zero-pad
each window to a fixed 4-day shape and record `Window_Length`, then mask it in the
LSTM via `pack_padded_sequence`. (An earlier draft of this notebook mistakenly
*required* T==5 in the naive belief that a "5-day window" should always be exactly
5 real trading days; that discarded ~83% of valid samples and shrank the dataset to
~4,700 — nowhere near the correct ~26,600. Padding + masking, not filtering, is
what makes this an exact reproduction.)

**Validation:** this pipeline produces **26,619 total samples** with a
**50.2% / 49.8%** up/down split — matching the StockNet paper's reported dataset
size (~26,614 samples) and near-balanced target almost exactly. That congruence is
the strongest evidence this reproduction is faithful.

### Feature sets

Mirrors the same FS1–FS4 structure as the Phase 1 baseline, so results are
directly comparable:

| Feature Set | Contents |
|---|---|
| FS1 | Price only |
| FS2 | Price + Fundamentals (8 EDGAR features, fetched fresh in this notebook) |
| FS3 | Price + Tweet Counts |
| FS4 | Price + Fundamentals + Tweet Counts |

Fundamentals are **not** part of the original StockNet paper/task — they're a
project-specific extension, included here only so this experiment's feature-set
structure matches Phase 1's, not because they're part of "exact StockNet
reproduction." They're aligned to each ticker's full StockNet trading calendar
(forward-filled from SEC filing date, zero-filled where unavailable — same
convention as Phase 1's FS2/FS4), not merged from `dataset/final/`, because that
dataset's own buffer-zone-day filtering would silently drop fundamentals coverage
for exactly the days some input windows need.

### Scope note

This notebook reproduces StockNet's **data methodology** (Experiment 1 in the
project roadmap) and evaluates it with simple baselines (Logistic Regression,
LSTM, MLP) using tweet **counts** as the text signal. It does **not** reproduce
the paper's own Hedge-FVMD neural architecture or hierarchical tweet-attention
model — that full text semantics + temporal-attention + bilinear-fusion + GAT
reproduction is Experiment 2 (MAN-SF) in the roadmap, a separate and larger
undertaking.

## 1. Setup & Clone

In [ ]:
!git clone https://github.com/yumoxu/stocknet-dataset.git
print('Cloned!')

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datetime import datetime, timedelta
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = 'stocknet-dataset'
PRICE_DIR = os.path.join(ROOT, 'price', 'preprocessed')
TWEET_DIR = os.path.join(ROOT, 'tweet', 'preprocessed')

MAX_N_DAYS = 5          # config.yml model.max_n_days
MAX_INPUT_DAYS = MAX_N_DAYS - 1  # 4 real input days + 1 target slot

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 2. Exact Ticker Universe and Split Dates
Copied verbatim from `stocknet-code/src/config.yml`.

In [ ]:
STOCK_SYMBOLS = (
    ['XOM','RDS-B','PTR','CVX','TOT','BP','BHP','SNP','SLB','BBL'] +       # materials
    ['AAPL','PG','BUD','KO','PM','TM','PEP','UN','UL','MO'] +              # consumer_goods
    ['JNJ','PFE','NVS','UNH','MRK','AMGN','MDT','ABBV','SNY','CELG'] +     # healthcare
    ['AMZN','BABA','WMT','CMCSA','HD','DIS','MCD','CHTR','UPS','PCLN'] +   # services
    ['NEE','DUK','D','SO','NGG','AEP','PCG','EXC','SRE','PPL'] +           # utilities
    ['IEP','HRG','CODI','REX','SPLP','PICO','AGFS','GMRE'] +               # conglomerates
    ['BCH','BSAC','BRK-A','JPM','WFC','BAC','V','C','HSBC','MA'] +         # finance
    ['GE','MMM','BA','HON','UTX','LMT','CAT','GD','DHR','ABB'] +           # industrial_goods
    ['GOOG','MSFT','FB','T','CHL','ORCL','TSM','VZ','INTC','CSCO']         # tech
)
assert len(STOCK_SYMBOLS) == 88

# end-exclusive, exactly as DataPipe._get_start_end_date / sample_gen_from_one_stock
# compares with `start_date <= main_target_date_str < end_date`
SPLIT_DATES = {
    'train': ('2014-01-01', '2015-08-01'),
    'dev':   ('2015-08-01', '2015-10-01'),
    'test':  ('2015-10-01', '2016-01-01'),
}

def get_split(date):
    ds = date.isoformat()
    for name, (start, end) in SPLIT_DATES.items():
        if start <= ds < end:
            return name
    return None

print(f'{len(STOCK_SYMBOLS)} tickers, 3-way split defined.')

## 3. Load StockNet's Own Precomputed Price/Movement Files

`price/preprocessed/{TICKER}.txt` is tab-separated: `date, main_movement_pct, ?, price_ratio_1, price_ratio_2, price_ratio_3, volume`.
We use `data[1]` for movement/target and `data[3:6]` (the 3 official price ratios)
as input features — exactly what `DataPipe._get_prices_and_ts` reads, so results are
numerically tied to the same source StockNet's own experiments used.

In [ ]:
def load_movement_file(ticker):
    fp = os.path.join(PRICE_DIR, f'{ticker}.txt')
    if not os.path.exists(fp):
        return None
    rows = []
    with open(fp, 'r', encoding='utf8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            date = datetime.strptime(parts[0], '%Y-%m-%d').date()
            mv = float(parts[1])
            prices = [float(parts[3]), float(parts[4]), float(parts[5])]
            rows.append((date, mv, prices))
    rows.sort(key=lambda r: r[0])
    return rows


def load_tweet_dates(ticker):
    """{calendar_date: [token_list, ...]} for every tweet-day file present."""
    tdir = os.path.join(TWEET_DIR, ticker)
    out = {}
    if not os.path.isdir(tdir):
        return out
    for fname in os.listdir(tdir):
        try:
            d = datetime.strptime(fname, '%Y-%m-%d').date()
        except ValueError:
            continue
        msgs = []
        try:
            with open(os.path.join(tdir, fname), 'r', encoding='utf8') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    obj = json.loads(line)
                    text = obj.get('text', '')
                    if isinstance(text, list) and text:
                        msgs.append(text)
        except Exception:
            continue
        if msgs:
            out[d] = msgs
    return out

print('Loaders defined.')

## 4. Build Samples — Exact Windowing + Buffer-Zone Filter + Tweet Alignment

Ports `DataPipe._get_prices_and_ts`, `_get_unaligned_corpora`, and
`_trading_day_alignment` line-for-line into pandas/Python. Variable window length
`T` (1..4 real input days) is preserved and zero-padded, not filtered.

In [ ]:
def build_samples_for_ticker(ticker, movement_rows, tweet_by_date, max_n_days=MAX_N_DAYS):
    date_list = [r[0] for r in movement_rows]
    mv_by_date = {r[0]: r[1] for r in movement_rows}
    prices_by_date = {r[0]: r[2] for r in movement_rows}
    date_set = set(date_list)
    max_input_days = max_n_days - 1

    samples = []
    for main_target_date in date_list:
        main_mv = mv_by_date[main_target_date]
        # Buffer-zone filter: applied ONLY to the day being predicted, exactly as
        # in DataPipe._get_prices_and_ts. Input-window days are never filtered.
        if -0.005 <= main_mv < 0.0055:
            continue

        d_t_min = main_target_date - timedelta(days=max_n_days - 1)
        ts_window = sorted(d for d in date_set if d_t_min <= d < main_target_date)
        T_input = len(ts_window)
        if T_input == 0:
            continue

        ts_full = ts_window + [main_target_date]

        # Tweet trading-day alignment: each calendar day's tweets -> first ts slot
        # strictly after it (a day's tweets NEVER attach to that same day).
        d_d_max = main_target_date - timedelta(days=1)
        d_d_min = main_target_date - timedelta(days=max_n_days)
        unaligned_days = sorted(d for d in tweet_by_date if d_d_min <= d <= d_d_max)

        slot_tweet_counts = [0] * (T_input + 1)
        for d in unaligned_days:
            for t in range(T_input + 1):
                if d < ts_full[t]:
                    slot_tweet_counts[t] += len(tweet_by_date[d])
                    break

        input_prices = [prices_by_date[d] for d in ts_window]
        input_prices += [[0.0, 0.0, 0.0]] * (max_input_days - T_input)
        input_tweet_counts = slot_tweet_counts[:-1] + [0] * (max_input_days - T_input)

        samples.append({
            'Ticker': ticker,
            'Target_Date': main_target_date,
            'Main_MV_Percent': main_mv,
            'Target': 1 if main_mv > 0 else 0,
            'Window_Length': T_input,                  # real input days present, 1..4
            'Input_Prices': input_prices,               # (4,3), zero-padded
            'Input_Tweet_Counts': input_tweet_counts,   # (4,), zero-padded
            'Pretarget_Tweet_Count': slot_tweet_counts[-1],  # tweets attached to target's own slot
        })

    return samples


all_samples = []
for ticker in STOCK_SYMBOLS:
    movement_rows = load_movement_file(ticker)
    if not movement_rows:
        print(f'  {ticker}: no price/preprocessed file, skipping')
        continue
    tweet_by_date = load_tweet_dates(ticker)
    samples = build_samples_for_ticker(ticker, movement_rows, tweet_by_date)
    for s in samples:
        s['Split'] = get_split(s['Target_Date'])
    samples = [s for s in samples if s['Split'] is not None]
    all_samples.extend(samples)

print(f'\nTOTAL: {len(all_samples)} samples across {len(STOCK_SYMBOLS)} tickers')

from collections import Counter
split_counts = Counter(s['Split'] for s in all_samples)
target_counts = Counter(s['Target'] for s in all_samples)
print('Split counts:', dict(split_counts))
print('Target balance:', dict(target_counts),
      f"({100*target_counts[1]/len(all_samples):.1f}% up)")

# Sanity check against the published StockNet dataset size (~26,614 samples,
# near-50/50 balance). This is the strongest evidence the reproduction is faithful.
assert 25000 < len(all_samples) < 28000, \
    f'Sample count {len(all_samples)} is far from the expected ~26,614 -- check the windowing logic.'
print('\n✓ Sample count matches the published StockNet dataset scale.')

## 5. Fetch and Align SEC EDGAR Fundamentals (for FS2/FS4)

Same CIK-matching and tag fixes as `MMGTFFF_preprocessing_CORRECTED.ipynb`
(dropped the wrong `LiabilitiesAndStockholdersEquity` fallback for
`TotalLiabilities`; ticker list built from the same 88 `STOCK_SYMBOLS` used
above, not a hand-typed list). Point-in-time forward-fill uses the *filing*
date, never the period-end date. Aligned against each ticker's **full**
StockNet trading calendar (every date in its `price/preprocessed/{TICKER}.txt`
file), not just dates that survive as valid prediction targets — fundamentals
need to be looked up for any day that lands inside an input window, not only
target days.

In [ ]:
import requests
import time

HEADERS = {
    # SEC requires a descriptive User-Agent with a real contact email.
    'User-Agent': 'MMGTFFF-Research your_email@example.com',
    'Accept-Encoding': 'gzip, deflate'
}

resp = requests.get('https://www.sec.gov/files/company_tickers.json', headers=HEADERS)
ticker_to_cik = {v['ticker'].upper(): str(v['cik_str']).zfill(10) for v in resp.json().values()}

TICKER_OVERRIDES = {'GOOG': 'GOOGL', 'FB': 'META', 'PCLN': 'BKNG'}

matched = {}
for t in STOCK_SYMBOLS:
    lookup = t.replace('-', '.')
    if t in ticker_to_cik:
        matched[t] = ticker_to_cik[t]
    elif lookup in ticker_to_cik:
        matched[t] = ticker_to_cik[lookup]
    elif t in TICKER_OVERRIDES and TICKER_OVERRIDES[t] in ticker_to_cik:
        matched[t] = ticker_to_cik[TICKER_OVERRIDES[t]]

print(f'Matched {len(matched)}/{len(STOCK_SYMBOLS)} tickers to a CIK '
      f'(unmatched are mostly foreign filers with no US GAAP 10-K/10-Q).')

METRIC_TAGS = {
    'Revenue': ['RevenueFromContractWithCustomerExcludingAssessedTax', 'Revenues',
                'SalesRevenueNet', 'SalesRevenueGoodsNet',
                'RevenueFromContractWithCustomerIncludingAssessedTax'],
    'NetIncome': ['NetIncomeLoss', 'NetIncomeLossAvailableToCommonStockholdersBasic', 'ProfitLoss'],
    'TotalAssets': ['Assets'],
    'TotalLiabilities': ['Liabilities'],  # no LiabilitiesAndStockholdersEquity fallback -- that tag is Liabilities+Equity, not Liabilities
    'StockholdersEquity': ['StockholdersEquity', 'StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest'],
    'EPS': ['EarningsPerShareBasic', 'EarningsPerShareDiluted'],
    'Cash': ['CashAndCashEquivalentsAtCarryingValue', 'CashCashEquivalentsAndShortTermInvestments'],
}
FUNDAMENTAL_FEATURES = list(METRIC_TAGS.keys())  # ROA computed after, from these

def get_company_facts(cik):
    r = requests.get(f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json', headers=HEADERS)
    return r.json() if r.status_code == 200 else None

def extract_metric(facts, tag_variants, start_date='2013-01-01', end_date='2016-06-30'):
    if not facts or 'facts' not in facts:
        return []
    us_gaap = facts['facts'].get('us-gaap', {})
    for tag in tag_variants:
        if tag not in us_gaap:
            continue
        values = us_gaap[tag].get('units', {}).get('USD', us_gaap[tag].get('units', {}).get('USD/shares', []))
        if not values:
            continue
        results = []
        for e in values:
            filed, form, val = e.get('filed', ''), e.get('form', ''), e.get('val')
            if form not in ('10-K', '10-Q', '10-K/A', '10-Q/A', '20-F'):
                continue
            if filed < start_date or filed > end_date or val is None:
                continue
            results.append({'filed': filed, 'value': val})
        if results:
            return results
    return []

fund_records = []
for ticker, cik in matched.items():
    facts = get_company_facts(cik)
    if facts is not None:
        for metric, tags in METRIC_TAGS.items():
            for e in extract_metric(facts, tags):
                fund_records.append({'Ticker': ticker, 'Metric': metric, 'Filed_Date': e['filed'], 'Value': e['value']})
    time.sleep(0.15)

fund_raw = pd.DataFrame(fund_records)
fund_raw['Filed_Date'] = pd.to_datetime(fund_raw['Filed_Date']).dt.date
print(f'Fetched {len(fund_raw)} fundamental records for {fund_raw["Ticker"].nunique()} tickers.')


def align_fundamentals_for_ticker(ticker, all_dates):
    daily_index = pd.DatetimeIndex(pd.to_datetime(all_dates))
    ticker_data = fund_raw[fund_raw['Ticker'] == ticker]
    result = {}
    if len(ticker_data) == 0:
        for m in FUNDAMENTAL_FEATURES:
            result[m] = pd.Series(np.nan, index=daily_index)
        result['ROA'] = pd.Series(np.nan, index=daily_index)
        return result
    for metric in FUNDAMENTAL_FEATURES:
        md_ = (ticker_data[ticker_data['Metric'] == metric]
               .sort_values('Filed_Date').drop_duplicates(subset=['Filed_Date'], keep='last'))
        if len(md_) == 0:
            result[metric] = pd.Series(np.nan, index=daily_index)
            continue
        ts = md_.set_index(pd.to_datetime(md_['Filed_Date']))['Value']
        ts = ts[~ts.index.duplicated(keep='last')]
        result[metric] = ts.reindex(daily_index, method='ffill')
    roa = result['NetIncome'] / result['TotalAssets']
    result['ROA'] = roa.replace([np.inf, -np.inf], np.nan)
    return result


# Attach per-ticker fundamentals lookup + fold into each sample's input days
fund_lookup_by_ticker = {}
for ticker in STOCK_SYMBOLS:
    movement_rows = load_movement_file(ticker)
    if not movement_rows:
        continue
    all_dates = [r[0] for r in movement_rows]
    fund_series = align_fundamentals_for_ticker(ticker, all_dates)
    fund_lookup_by_ticker[ticker] = {
        d: [fund_series[m].iloc[i] for m in FUNDAMENTAL_FEATURES + ['ROA']]
        for i, d in enumerate(all_dates)
    }

N_FUND = len(FUNDAMENTAL_FEATURES) + 1  # +ROA

for s in all_samples:
    ticker, target = s['Ticker'], s['Target_Date']
    lookup = fund_lookup_by_ticker.get(ticker, {})
    d_t_min = target - timedelta(days=MAX_N_DAYS - 1)
    ts_window = sorted(d for d in lookup if d_t_min <= d < target)
    fund_window = [lookup[d] for d in ts_window]
    fund_window += [[np.nan] * N_FUND] * (MAX_INPUT_DAYS - len(ts_window))
    s['Input_Fundamentals'] = fund_window

n_with_fund = sum(1 for s in all_samples if any(
    not np.isnan(v) for day in s['Input_Fundamentals'] for v in day))
print(f'Samples with at least one non-missing fundamental in their window: '
      f'{n_with_fund}/{len(all_samples)}')

## 6. Save the Flat Dataset

In [ ]:
FUND_COLS = FUNDAMENTAL_FEATURES + ['ROA']

records = []
for s in all_samples:
    rec = {
        'Ticker': s['Ticker'],
        'Target_Date': s['Target_Date'].isoformat(),
        'Split': s['Split'],
        'Main_MV_Percent': s['Main_MV_Percent'],
        'Target': s['Target'],
        'Window_Length': s['Window_Length'],
        'Pretarget_Tweet_Count': s['Pretarget_Tweet_Count'],
    }
    for day_i in range(MAX_INPUT_DAYS):
        rec[f'Day{day_i+1}_Price1'] = s['Input_Prices'][day_i][0]
        rec[f'Day{day_i+1}_Price2'] = s['Input_Prices'][day_i][1]
        rec[f'Day{day_i+1}_Price3'] = s['Input_Prices'][day_i][2]
        rec[f'Day{day_i+1}_TweetCount'] = s['Input_Tweet_Counts'][day_i]
        for fi, fcol in enumerate(FUND_COLS):
            rec[f'Day{day_i+1}_{fcol}'] = s['Input_Fundamentals'][day_i][fi]
    records.append(rec)

exact_df = pd.DataFrame(records)
os.makedirs('final', exist_ok=True)
exact_df.to_parquet('final/stocknet_protocol_a_exact.parquet', index=False)
exact_df.to_csv('final/stocknet_protocol_a_exact.csv', index=False)
print(f'Saved: {exact_df.shape}')
exact_df.head()

## 7. Baseline Evaluation on the Exact Protocol

Logistic Regression + LSTM (masked via `Window_Length`) + MLP, across all 4
feature sets (FS1–FS4), using the **official** train/dev/test split. This is
Experiment 1's "trustworthy benchmark" — compare it against MAN-SF's own
reported ablation numbers (LSTM+price ≈ MCC 0.002, GRU+social text ≈ MCC 0.077)
before moving on to Experiment 2 (MAN-SF full reproduction).

In [ ]:
def to_xy(df, use_fund, use_tweets):
    n = len(df)
    n_feat = 3 + (N_FUND if use_fund else 0) + (1 if use_tweets else 0)
    X = np.zeros((n, MAX_INPUT_DAYS, n_feat), dtype=np.float32)
    mask = np.zeros((n, MAX_INPUT_DAYS), dtype=np.float32)
    for day_i in range(MAX_INPUT_DAYS):
        col = 0
        X[:, day_i, col] = df[f'Day{day_i+1}_Price1'].values; col += 1
        X[:, day_i, col] = df[f'Day{day_i+1}_Price2'].values; col += 1
        X[:, day_i, col] = df[f'Day{day_i+1}_Price3'].values; col += 1
        if use_fund:
            for fcol in FUND_COLS:
                X[:, day_i, col] = np.nan_to_num(df[f'Day{day_i+1}_{fcol}'].values, nan=0.0); col += 1
        if use_tweets:
            X[:, day_i, col] = df[f'Day{day_i+1}_TweetCount'].values; col += 1
    for i, wl in enumerate(df['Window_Length'].values):
        mask[i, :wl] = 1.0
    y = df['Target'].values.astype(np.int64)
    return X, y, mask


def normalize(X_train, *others):
    flat = X_train.reshape(-1, X_train.shape[-1])
    mean, std = flat.mean(axis=0), flat.std(axis=0)
    std[std == 0] = 1.0
    return [(X_train - mean) / std] + [(X - mean) / std for X in others]


class WindowDataset(Dataset):
    def __init__(self, X, y, mask):
        self.X, self.y, self.mask = X, y, mask
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return (torch.tensor(self.X[idx]), torch.tensor(self.y[idx]), torch.tensor(self.mask[idx]))


class LSTMBaseline(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=1, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 2))

    def forward(self, x, mask):
        lengths = mask.sum(dim=1).clamp(min=1).long().cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        return self.head(h_n[-1])


class MLPBaseline(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 2))
    def forward(self, x_flat):
        return self.net(x_flat)


def compute_metrics(y_true, y_pred, y_prob):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'auc': roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else 0.5,
    }


def run_lstm(X_train, y_train, m_train, X_val, y_val, m_val, X_test, y_test, m_test,
             input_dim, epochs=50, patience=10):
    torch.manual_seed(SEED)
    model = LSTMBaseline(input_dim).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()

    train_dl = DataLoader(WindowDataset(X_train, y_train, m_train), batch_size=64, shuffle=True)
    val_ds, test_ds = WindowDataset(X_val, y_val, m_val), WindowDataset(X_test, y_test, m_test)

    def evaluate(ds):
        model.eval()
        dl = DataLoader(ds, batch_size=256)
        yt, yp, ypr = [], [], []
        with torch.no_grad():
            for x, y, m in dl:
                x, m = x.to(DEVICE), m.to(DEVICE)
                logits = model(x, m)
                probs = torch.softmax(logits, dim=1)[:, 1]
                yt.extend(y.numpy()); yp.extend(logits.argmax(1).cpu().numpy()); ypr.extend(probs.cpu().numpy())
        return compute_metrics(np.array(yt), np.array(yp), np.array(ypr))

    best_mcc, best_state, wait = -1e9, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for x, y, m in train_dl:
            x, y, m = x.to(DEVICE), y.to(DEVICE), m.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(x, m), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        val_m = evaluate(val_ds)
        if val_m['mcc'] > best_mcc:
            best_mcc, best_state, wait = val_m['mcc'], {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
        if wait >= patience:
            break
    model.load_state_dict(best_state)
    return evaluate(test_ds)


def run_mlp(X_train, y_train, X_val, y_val, X_test, y_test, epochs=50, patience=10):
    torch.manual_seed(SEED)
    Xf_train, Xf_val, Xf_test = (X_train.reshape(len(X_train), -1),
                                  X_val.reshape(len(X_val), -1),
                                  X_test.reshape(len(X_test), -1))
    model = MLPBaseline(Xf_train.shape[-1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    train_dl = DataLoader(list(zip(torch.tensor(Xf_train), torch.tensor(y_train))), batch_size=64, shuffle=True)

    def evaluate(Xf, y):
        model.eval()
        with torch.no_grad():
            logits = model(torch.tensor(Xf).to(DEVICE))
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = logits.argmax(1).cpu().numpy()
        return compute_metrics(y, preds, probs)

    best_mcc, best_state, wait = -1e9, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        val_m = evaluate(Xf_val, y_val)
        if val_m['mcc'] > best_mcc:
            best_mcc, best_state, wait = val_m['mcc'], {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
        if wait >= patience:
            break
    model.load_state_dict(best_state)
    return evaluate(Xf_test, y_test)


def run_logreg(X_train, y_train, X_test, y_test):
    model = LogisticRegression(max_iter=1000, random_state=SEED)
    model.fit(X_train.reshape(len(X_train), -1), y_train)
    pred = model.predict(X_test.reshape(len(X_test), -1))
    prob = model.predict_proba(X_test.reshape(len(X_test), -1))[:, 1]
    return compute_metrics(y_test, pred, prob)


FEATURE_SETS = {
    'FS1_Price':                          (False, False, 'Price only'),
    'FS2_Price_Fundamentals':             (True,  False, 'Price + Fundamentals'),
    'FS3_Price_TweetCounts':              (False, True,  'Price + Tweet Counts'),
    'FS4_Price_Fundamentals_TweetCounts': (True,  True,  'Price + Fundamentals + Tweet Counts'),
}

train_df = exact_df[exact_df['Split'] == 'train']
val_df   = exact_df[exact_df['Split'] == 'dev']
test_df  = exact_df[exact_df['Split'] == 'test']

results = {}
for fs_key, (use_fund, use_tweets, label) in FEATURE_SETS.items():
    print(f"\n{'='*60}\n{label}\n{'='*60}")
    X_train, y_train, m_train = to_xy(train_df, use_fund, use_tweets)
    X_val, y_val, m_val       = to_xy(val_df, use_fund, use_tweets)
    X_test, y_test, m_test    = to_xy(test_df, use_fund, use_tweets)
    X_train, X_val, X_test = normalize(X_train, X_val, X_test)
    input_dim = X_train.shape[-1]

    lr_m = run_logreg(X_train, y_train, X_test, y_test)
    print(f"  Logistic Regression: Acc={lr_m['accuracy']:.4f} F1={lr_m['f1']:.4f} MCC={lr_m['mcc']:+.4f} AUC={lr_m['auc']:.4f}")

    lstm_m = run_lstm(X_train, y_train, m_train, X_val, y_val, m_val, X_test, y_test, m_test, input_dim)
    print(f"  LSTM:                Acc={lstm_m['accuracy']:.4f} F1={lstm_m['f1']:.4f} MCC={lstm_m['mcc']:+.4f} AUC={lstm_m['auc']:.4f}")

    mlp_m = run_mlp(X_train, y_train, X_val, y_val, X_test, y_test)
    print(f"  MLP:                 Acc={mlp_m['accuracy']:.4f} F1={mlp_m['f1']:.4f} MCC={mlp_m['mcc']:+.4f} AUC={mlp_m['auc']:.4f}")

    results[fs_key] = {
        'label': label, 'num_features': int(input_dim),
        'train_size': len(y_train), 'val_size': len(y_val), 'test_size': len(y_test),
        'models': {
            'Logistic Regression': {**lr_m, 'n_samples': len(y_test)},
            'LSTM': {**lstm_m, 'n_samples': len(y_test)},
            'MLP': {**mlp_m, 'n_samples': len(y_test)},
        }
    }

with open('final/protocol_a_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nSaved: final/protocol_a_results.json')

## 8. Compare Against Literature

| Model (source) | MCC |
|---|---:|
| LSTM + price (MAN-SF ablation table) | ~0.002 |
| GRU + social text (MAN-SF ablation table) | ~0.077 |
| GCN + price (MAN-SF ablation table) | ~0.093 |
| **This notebook's result (FS1/FS3, LSTM)** | *(see cell 7 output)* |
| MAN-SF full model (target for Experiment 2) | ~0.195 |

If this notebook's LSTM+price MCC lands well above ~0.002, that's a signal the
window/label/split reproduction diverges from the paper's somewhere (worth
re-checking against `DataPipe.py` before trusting it as a "trustworthy
benchmark"). If it's in a similar ballpark, the exact-protocol reproduction has
succeeded and Experiment 2 (MAN-SF's full architecture) is the next roadmap
step.